# 🚀 Ollama en Google Colab con GPU T4 + Ngrok
### Proyecto: **Planos Históricos Organización**

Este notebook levanta **Ollama** con aceleración de GPU (NVIDIA T4 gratuita de Colab) y expone la API mediante un túnel de **Ngrok** para que el backend local (`fastapi` / `docker-compose`) procese planos de forma ultrarrápida (3 a 5 segundos por plano).

---  
### ⚠️ Requisito previo importante:
Verifica que este notebook esté corriendo con **GPU**:
👉 Menú superior: **Entorno de ejecución** -> **Cambiar tipo de entorno de ejecución** -> Seleccionar **GPU T4**.

In [ ]:
#@title 📦 1. Instalar dependencias, Ollama y pyngrok
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q pyngrok

---  
### 🔑 2. Iniciar Ollama y Túnel Ngrok
1. Ve a tu panel de Ngrok: [https://dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken)
2. Copia tu **Authtoken** y pégalo en el campo `NGROK_TOKEN` a continuación.
3. Ejecuta la celda.

In [ ]:
#@title ⚙️ Iniciar Ollama y generar URL pública
!pip install -q pyngrok
import os
import subprocess
import time
from pyngrok import ngrok

NGROK_TOKEN = "TU_AUTHTOKEN_AQUI" #@param {type:"string"}

if NGROK_TOKEN == "TU_AUTHTOKEN_AQUI" or not NGROK_TOKEN.strip():
    raise ValueError("⚠️ Debes ingresar tu authtoken de ngrok antes de continuar.")

# 1. Configurar token en ngrok y limpiar túneles previos
ngrok.kill()
ngrok.set_auth_token(NGROK_TOKEN.strip())

# 2. Permitir conexiones a Ollama desde cualquier IP y origen
os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
os.environ["OLLAMA_ORIGINS"] = "*"

# 3. Terminar instancias previas si existieran
subprocess.run(["pkill", "-f", "ollama serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)

# 4. Iniciar demonio de Ollama en segundo plano guardando logs en /content/ollama.log
print("Iniciando servicio de Ollama en background (logs en /content/ollama.log)...")
log_file = open("/content/ollama.log", "a")
ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=log_file, stderr=subprocess.STDOUT)
time.sleep(3)

# 5. Abrir túnel ngrok hacia el puerto 11434 reescribiendo el Host header
tunnel = ngrok.connect(11434, "http", host_header="localhost:11434")
public_url = tunnel.public_url

print("\n" + "=" * 65)
print("🚀 ¡OLLAMA ESTÁ CORRIENDO EN GOOGLE COLAB!")
print(f"Copia esta URL en tu archivo .env local:")
print(f"OLLAMA_HOST={public_url}")
print("=" * 65)

---  
### 📥 3. Descargar el modelo de visión
Por defecto el proyecto usa `qwen2.5vl:3b`. Si deseas experimentar con mayor precisión en sellos difíciles, puedes elegir `qwen2.5vl:7b` (la GPU T4 tiene 16 GB VRAM y lo soporta sin problemas).

In [ ]:
#@title 📥 Descargar modelo
MODELO = "qwen2.5vl:3b" #@param ["qwen2.5vl:3b", "qwen2.5vl:7b"]
!ollama pull {MODELO}

---  
### 🩺 4. Verificar que el servicio responde a través del túnel

In [ ]:
#@title 🩺 Test de salud del túnel y modelos disponibles
import requests

try:
    res = requests.get(f"{public_url}/api/tags", headers={"ngrok-skip-browser-warning": "true"}, timeout=10)
    res.raise_for_status()
    modelos = [m["name"] for m in res.json().get("models", [])]
    print(f"✅ Túnel activo y funcionando correctamente!")
    print(f"Modelos listos en Colab: {modelos}")
except Exception as e:
    print(f"❌ Error al verificar túnel: {e}")

---  
### 📊 5. Monitoreo de GPU (opcional)
Puedes ejecutar esta celda en cualquier momento para ver el consumo de VRAM y el estado de la GPU NVIDIA.

In [ ]:
!nvidia-smi

---  
### ⏱️ 6. Mantener la sesión activa mientras usas el programa local
Ejecuta esta celda para mantener el notebook corriendo indefinidamente mientras pruebas la API o la aplicación web local.

In [ ]:
#@title 📜 6. Ver logs de Ollama en vivo (y mantener sesión activa)
!tail -f /content/ollama.log